# Taller 4 - Entrenamiento Multi-Modelo (Penguins)

Notebook que:
1. Carga datos desde PostgreSQL (dataset Penguins)
2. Procesa y guarda en data_procesada
3. Entrena **6 técnicas de ML** (RandomForest, LogisticRegression, DecisionTree, GradientBoosting, KNN, SVM)
4. **20 iteraciones por técnica** con diferentes hiperparámetros
5. Registra todo en MLflow

## 1. Configuración y carga de datos

In [1]:
import os
import pandas as pd
from sqlalchemy import create_engine
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature

PG_HOST = os.environ.get("POSTGRES_HOST", "postgres")
PG_PORT = os.environ.get("POSTGRES_PORT", "5432")
PG_USER = os.environ.get("POSTGRES_USER", "admin")  # O el usuario que hayas dejado
PG_PASS = os.environ.get("POSTGRES_PASSWORD", "admin")
PG_DB = os.environ.get("POSTGRES_DB", "data_db") # Leemos la variable de entorno que pusimos en el compose
CONN_RAW = f"postgresql://{PG_USER}:{PG_PASS}@{PG_HOST}:{PG_PORT}/{PG_DB}"

mlflow.set_tracking_uri(os.environ.get("MLFLOW_TRACKING_URI", "http://mlflow:5000"))
try:
    exp = mlflow.get_experiment_by_name("penguins-multimodel")
    if exp and "s3://" in (exp.artifact_location or ""):
        mlflow.delete_experiment(exp.experiment_id)
except Exception:
    pass
mlflow.set_experiment("penguins-multimodel")

2026/03/21 19:11:34 INFO mlflow.tracking.fluent: Experiment with name 'penguins-multimodel' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/2', creation_time=1774120294657, experiment_id='2', last_update_time=1774120294657, lifecycle_stage='active', name='penguins-multimodel', tags={}>

In [2]:
engine = create_engine(CONN_RAW)
try:
    df_raw = pd.read_sql("SELECT * FROM data_raw", con=engine)
    if len(df_raw) == 0:
        raise ValueError("Tabla vacía")
except Exception:
    csv_path = "/home/jovyan/data/penguins.csv"
    df_raw = pd.read_csv(csv_path)
    df_raw.to_sql("data_raw", engine, if_exists="replace", index=False)
    print(f"Cargados {len(df_raw)} registros desde CSV")
else:
    print(f"Registros en data_raw: {len(df_raw)}")

Registros en data_raw: 344


## 2. Procesamiento

In [3]:
df = df_raw.dropna().copy()
le_island = LabelEncoder()
le_sex = LabelEncoder()
le_species = LabelEncoder()
df["island_encoded"] = le_island.fit_transform(df["island"].astype(str))
df["sex_encoded"] = le_sex.fit_transform(df["sex"].astype(str))
df["species_encoded"] = le_species.fit_transform(df["species"].astype(str))

cols_procesada = [
    "bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g",
    "island_encoded", "sex_encoded", "year", "species_encoded"
]
df_proc = df[cols_procesada]
df_proc.to_sql("data_procesada", engine, if_exists="replace", index=False)
print(f"Guardados {len(df_proc)} registros en data_procesada")

Guardados 333 registros en data_procesada


In [4]:
X = df_proc.drop("species_encoded", axis=1)
y = df_proc["species_encoded"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# LR, KNN, SVM usan Pipeline con StandardScaler internamente

## 3. Entrenamiento Multi-Modelo (6 técnicas × 20 iteraciones)

In [5]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score
import numpy as np
import itertools

In [6]:
def train_and_log(model, params, X_tr, X_te, y_tr, y_te, model_name):
    """Entrena, evalúa y registra en MLflow."""
    model.fit(X_tr, y_tr)
    pred = model.predict(X_te)
    acc = accuracy_score(y_te, pred)
    f1 = f1_score(y_te, pred, average="weighted")
    for k, v in params.items():
        mlflow.log_param(k, v)
    mlflow.log_metric("accuracy", acc)
    mlflow.log_metric("f1_weighted", f1)
    mlflow.set_tag("model_type", model_name)
    sig = infer_signature(X_tr, model.predict(X_tr))
    mlflow.sklearn.log_model(model, "model", registered_model_name=model_name, signature=sig)
    return acc

In [7]:
# 1. RandomForest - 20 combinaciones
rf_params = {
    "n_estimators": [50, 100, 150, 200],
    "max_depth": [5, 10, 15, 20, None],
    "min_samples_split": [2, 5]
}
rf_combos = list(itertools.product(rf_params["n_estimators"], rf_params["max_depth"], rf_params["min_samples_split"]))[:20]
for n_est, max_d, min_ss in rf_combos:
    with mlflow.start_run():
        m = RandomForestClassifier(n_estimators=n_est, max_depth=max_d, min_samples_split=min_ss, random_state=42)
        train_and_log(m, {"n_estimators": n_est, "max_depth": max_d or -1, "min_samples_split": min_ss},
                     X_train, X_test, y_train, y_test, "PenguinsRF")
print("RandomForest: 20 runs completados")

/opt/conda/lib/python3.11/site-packages/mlflow/types/utils.py:407: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
Registered model 'PenguinsRF' already exists. Creating a new version of this model...
2026/03/21 19:12:03 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: PenguinsRF, versi

RandomForest: 20 runs completados


In [8]:
# 2. LogisticRegression - 20 combinaciones
lr_params = {
    "C": [0.01, 0.1, 1.0, 10.0],
    "solver": ["lbfgs", "saga"],
    "max_iter": [500, 1000, 2000]
}
lr_combos = list(itertools.product(lr_params["C"], lr_params["solver"], lr_params["max_iter"]))[:20]
for c, solv, max_it in lr_combos:
    with mlflow.start_run():
        pipe = Pipeline([("scaler", StandardScaler()), ("clf", LogisticRegression(C=c, solver=solv, max_iter=max_it, random_state=42))])
        train_and_log(pipe, {"C": c, "solver": solv, "max_iter": max_it},
                     X_train, X_test, y_train, y_test, "PenguinsLR")
print("LogisticRegression: 20 runs completados")

/opt/conda/lib/python3.11/site-packages/mlflow/types/utils.py:407: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
Successfully registered model 'PenguinsLR'.
2026/03/21 19:12:53 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: PenguinsLR, version 1
Created version '1' of model 'Penguin

LogisticRegression: 20 runs completados


In [9]:
# 3. DecisionTree - 20 combinaciones
dt_params = {
    "max_depth": [3, 5, 7, 10, 15, 20, None],
    "min_samples_split": [2, 5, 10],
    "criterion": ["gini", "entropy"]
}
dt_combos = list(itertools.product(dt_params["max_depth"], dt_params["min_samples_split"], dt_params["criterion"]))[:20]
for max_d, min_ss, crit in dt_combos:
    with mlflow.start_run():
        m = DecisionTreeClassifier(max_depth=max_d, min_samples_split=min_ss, criterion=crit, random_state=42)
        train_and_log(m, {"max_depth": max_d or -1, "min_samples_split": min_ss, "criterion": crit},
                     X_train, X_test, y_train, y_test, "PenguinsDT")
print("DecisionTree: 20 runs completados")

/opt/conda/lib/python3.11/site-packages/mlflow/types/utils.py:407: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
Successfully registered model 'PenguinsDT'.
2026/03/21 19:14:16 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: PenguinsDT, version 1
Created version '1' of model 'Penguin

DecisionTree: 20 runs completados


In [10]:
# 4. GradientBoosting - 20 combinaciones
gb_params = {
    "n_estimators": [50, 100, 150, 200],
    "max_depth": [2, 3, 4, 5],
    "learning_rate": [0.01, 0.1, 0.2]
}
gb_combos = list(itertools.product(gb_params["n_estimators"], gb_params["max_depth"], gb_params["learning_rate"]))[:20]
for n_est, max_d, lr in gb_combos:
    with mlflow.start_run():
        m = GradientBoostingClassifier(n_estimators=n_est, max_depth=max_d, learning_rate=lr, random_state=42)
        train_and_log(m, {"n_estimators": n_est, "max_depth": max_d, "learning_rate": lr},
                     X_train, X_test, y_train, y_test, "PenguinsGB")
print("GradientBoosting: 20 runs completados")

/opt/conda/lib/python3.11/site-packages/mlflow/types/utils.py:407: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
Successfully registered model 'PenguinsGB'.
2026/03/18 11:47:49 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: PenguinsGB, version 1
Created version '1' of model 'Penguin

GradientBoosting: 20 runs completados


In [9]:
# 5. KNN - 20 combinaciones
knn_params = {
    "n_neighbors": [3, 5, 7, 10, 15, 20, 25, 30],
    "weights": ["uniform", "distance"],
    "p": [1, 2]
}
knn_combos = list(itertools.product(knn_params["n_neighbors"], knn_params["weights"], knn_params["p"]))[:20]
for k, w, p in knn_combos:
    with mlflow.start_run():
        pipe = Pipeline([("scaler", StandardScaler()), ("clf", KNeighborsClassifier(n_neighbors=k, weights=w, p=p))])
        train_and_log(pipe, {"n_neighbors": k, "weights": w, "p": p},
                     X_train, X_test, y_train, y_test, "PenguinsKNN")
print("KNN: 20 runs completados")

2026/03/21 17:26:13 INFO mlflow.tracking._tracking_service.client: 🏃 View run carefree-sloth-209 at: http://mlflow:5000/#/experiments/1/runs/1884e2897ec043e3977567f54222a363.
2026/03/21 17:26:13 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://mlflow:5000/#/experiments/1.


NameError: name 'train_and_log' is not defined

In [12]:
# 6. SVM - 20 combinaciones
svm_params = {
    "C": [0.1, 1.0, 10.0],
    "kernel": ["rbf", "linear", "poly"],
    "gamma": ["scale", "auto"]
}
svm_combos = list(itertools.product(svm_params["C"], svm_params["kernel"], svm_params["gamma"]))[:20]
for c, kern, gam in svm_combos:
    with mlflow.start_run():
        pipe = Pipeline([("scaler", StandardScaler()), ("clf", SVC(C=c, kernel=kern, gamma=gam, random_state=42))])
        train_and_log(pipe, {"C": c, "kernel": kern, "gamma": gam},
                     X_train, X_test, y_train, y_test, "PenguinsSVM")
print("SVM: 20 runs completados")

/opt/conda/lib/python3.11/site-packages/mlflow/types/utils.py:407: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
Successfully registered model 'PenguinsSVM'.
2026/03/18 11:48:40 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: PenguinsSVM, version 1
Created version '1' of model 'Pengu

SVM: 20 runs completados


## 4. Promover mejores modelos a Production

In [10]:
from mlflow.tracking import MlflowClient

client = MlflowClient()
exp = mlflow.get_experiment_by_name("penguins-multimodel")
model_names = ["PenguinsRF", "PenguinsLR", "PenguinsDT", "PenguinsGB", "PenguinsKNN", "PenguinsSVM"]

for model_name in model_names:
    runs = client.search_runs(
        experiment_ids=[exp.experiment_id],
        filter_string=f"tags.model_type = '{model_name}'",
        order_by=["metrics.accuracy DESC"]
    )
    if runs:
        best = runs[0]
        for mv in client.search_model_versions(f"run_id='{best.info.run_id}'"):
            client.transition_model_version_stage(model_name, mv.version, "Production")
            print(f"{model_name} v{mv.version} -> Production (acc={best.data.metrics.get('accuracy', 0):.3f})")
            break
print("Listo. Usa POST /predict/models con model_name para inferencia.")

Listo. Usa POST /predict/models con model_name para inferencia.
